In [2]:
import pandas as pd
import numpy as np
import re

In [1]:
EU_COUNTRIES = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

In [19]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s).strip().upper()
    match = re.fullmatch(r'([A-U])', s)

    if match:
        return match.group(1)
    else:
        return np.nan

## AI adoption (2021-2024 with gap year 2022)

In [ ]:
path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/raw data/'

ain2 = pd.read_csv(path + 'estat_isoc_eb_ain2.tsv/estat_isoc_eb_ain2.tsv', sep='\t')
print(ain2)

       freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD 2021    2023  2024 
0                      A,GE10,C,E_AIX_CC1SIX_DA,PC_ENT,AT     :    8.24     : 
1                      A,GE10,C,E_AIX_CC1SIX_DA,PC_ENT,BA     :   17.89     : 
2                      A,GE10,C,E_AIX_CC1SIX_DA,PC_ENT,BE     :   21.08     : 
3                      A,GE10,C,E_AIX_CC1SIX_DA,PC_ENT,BG     :   13.97     : 
4                      A,GE10,C,E_AIX_CC1SIX_DA,PC_ENT,CY     :    9.89     : 
...                                                   ...    ...     ...   ...
221987            A,GE10,S951,E_DI3_VLO_AI_TANY,PC_ENT,RO     :       0     : 
221988            A,GE10,S951,E_DI3_VLO_AI_TANY,PC_ENT,RS     :       0     : 
221989            A,GE10,S951,E_DI3_VLO_AI_TANY,PC_ENT,SE     :       0     : 
221990            A,GE10,S951,E_DI3_VLO_AI_TANY,PC_ENT,SI     :       0     : 
221991            A,GE10,S951,E_DI3_VLO_AI_TANY,PC_ENT,SK     :       0     : 

[221992 rows x 4 columns]


In [47]:
ain2_split = ain2[r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD'].str.split(',', expand=True)
ain2 = pd.concat([ain2, ain2_split], axis=1)
ain2.rename(columns={0: 'freq', 1: 'size_emp', 2: 'nace_r2',  3: 'indic_is', 4: 'unit', 5: 'geo_TIME_PERIOD'}, inplace=True)
ain2 = ain2.drop(r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD', axis=1)
ain2.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
ain2.columns = ain2.columns.str.strip()
ain2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221992 entries, 0 to 221991
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   2021      221992 non-null  object
 1   2023      221992 non-null  object
 2   2024      221992 non-null  object
 3   freq      221992 non-null  object
 4   size_emp  221992 non-null  object
 5   nace_r2   221992 non-null  object
 6   indic_is  221992 non-null  object
 7   unit      221992 non-null  object
 8   geo       221992 non-null  object
dtypes: object(9)
memory usage: 15.2+ MB


In [48]:
years_take = ['2021', '2023', '2024']

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = ain2[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    ain2[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    # flag_col_name = f'flag_{year}'
    # ain2[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
print(ain2)

       2021    2023 2024 freq size_emp nace_r2           indic_is    unit geo  \
0        :    8.24    :     A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  AT   
1        :   17.89    :     A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BA   
2        :   21.08    :     A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BE   
3        :   13.97    :     A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BG   
4        :    9.89    :     A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  CY   
...     ...     ...  ...  ...      ...     ...                ...     ...  ..   
221987   :       0    :     A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  RO   
221988   :       0    :     A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  RS   
221989   :       0    :     A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SE   
221990   :       0    :     A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SI   
221991   :       0    :     A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SK   

        num_2021  num_2023 

In [49]:
ain2.drop(['2021', '2023', '2024'], axis='columns', inplace=True)

In [50]:
ain2.describe()

,num_2021,num_2023,num_2024
count,105362.000000,156020.000000,132088.000000
mean,10.868867,14.625982,14.061412
std,20.984997,22.858054,20.959793
min,0.000000,0.000000,0.000000
25%,0.520000,1.210000,1.730000
50%,2.430000,4.000000,5.240000
75%,8.330000,16.290000,16.090000
max,100.000000,100.000000,100.000000


In [51]:
ain2.describe(include=["object", "bool"])

,freq,size_emp,nace_r2,indic_is,unit,geo
count,221992,221992,221992,221992,221992,221992
unique,1,1,48,62,6,36
top,A,GE10,C-E,E_AI_BCDP,PC_ENT,PL
freq,221992,221992,4717,5125,88814,7716


In [52]:
# Filter the data 
# E_AI_TANY - Enterprises use at least one of the AI technologies

ai_adopt = ain2.copy()

ai_adopt = ai_adopt[
    (ai_adopt["indic_is"] == "E_AI_TANY") &
    (ai_adopt["unit"] == "PC_ENT") 
]

ai_adopt = ai_adopt.drop(columns=['indic_is','unit',  'freq', 'size_emp'])

ai_adopt = ai_adopt[ai_adopt['geo'].isin(EU_EFTA)]
print(ai_adopt)

       nace_r2 geo  num_2021  num_2023  num_2024
3809         C  AT      9.61     12.31     22.71
3811         C  BE     10.42     15.31     23.24
3812         C  BG      2.88      2.55      4.35
3813         C  CY      2.05      3.81      2.98
3814         C  CZ      4.19      6.01      9.55
...        ...  ..       ...       ...       ...
221156    S951  PT      7.58      8.82     21.94
221157    S951  RO      0.00      0.00      3.08
221159    S951  SE      6.67      6.67       NaN
221160    S951  SI      0.00       NaN     19.22
221161    S951  SK      0.00      0.00      9.09

[1339 rows x 5 columns]


In [53]:
ai_adopt = ai_adopt.rename(columns={'num_2021' : 'ai_2021',
                                    'num_2023' : 'ai_2023',
                                    'num_2024' : 'ai_2024'})
ai_adopt['nace_r2_1d'] = ai_adopt['nace_r2'].map(nace_section_or_nan)
print(ai_adopt)

       nace_r2 geo  ai_2021  ai_2023  ai_2024 nace_r2_1d
3809         C  AT     9.61    12.31    22.71          C
3811         C  BE    10.42    15.31    23.24          C
3812         C  BG     2.88     2.55     4.35          C
3813         C  CY     2.05     3.81     2.98          C
3814         C  CZ     4.19     6.01     9.55          C
...        ...  ..      ...      ...      ...        ...
221156    S951  PT     7.58     8.82    21.94        NaN
221157    S951  RO     0.00     0.00     3.08        NaN
221159    S951  SE     6.67     6.67      NaN        NaN
221160    S951  SI     0.00      NaN    19.22        NaN
221161    S951  SK     0.00     0.00     9.09        NaN

[1339 rows x 6 columns]


In [54]:
ai_adopt = ai_adopt.dropna()
print(pd.unique(ai_adopt['nace_r2']))
print(ai_adopt)

['C' 'E' 'F' 'G' 'H' 'I' 'J' 'M' 'N']
       nace_r2 geo  ai_2021  ai_2023  ai_2024 nace_r2_1d
3809         C  AT     9.61    12.31    22.71          C
3811         C  BE    10.42    15.31    23.24          C
3812         C  BG     2.88     2.55     4.35          C
3813         C  CY     2.05     3.81     2.98          C
3814         C  CZ     4.19     6.01     9.55          C
...        ...  ..      ...      ...      ...        ...
207613       N  PT    10.81    10.29    12.91          N
207614       N  RO     3.04     1.57     5.30          N
207616       N  SE     8.58     8.30    21.63          N
207617       N  SI     9.02     3.22     7.65          N
207618       N  SK     8.60    10.16    18.37          N

[238 rows x 6 columns]


In [55]:
# Create the 2022 column by averaging 2021 and 2023
ai_adopt['ai_2022'] = (ai_adopt['ai_2021'] + ai_adopt['ai_2023']) / 2

# Check the results
print(ai_adopt[['ai_2021', 'ai_2022', 'ai_2023']].head())

      ai_2021  ai_2022  ai_2023
3809     9.61   10.960    12.31
3811    10.42   12.865    15.31
3812     2.88    2.715     2.55
3813     2.05    2.930     3.81
3814     4.19    5.100     6.01


In [157]:
ai_adopt = ai_adopt.drop(columns='nace_r2_1d')
cols_to_melt = ['ai_2021', 'ai_2022', 'ai_2023', 'ai_2024']

df_ai_panel = ai_adopt.melt(
    id_vars=['geo', 'nace_r2'],      
    value_vars=cols_to_melt,        
    var_name='year_raw',             
    value_name='ai_adoption'        
)


df_ai_panel['year'] = df_ai_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_ai_panel.drop(columns=['year_raw'], inplace=True)
df_ai_panel = df_ai_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print("AI Panel Format Ready:")
print(df_ai_panel.head(10))

AI Panel Format Ready:
  geo nace_r2  ai_adoption  year
0  AT       C        9.610  2021
1  AT       C       10.960  2022
2  AT       C       12.310  2023
3  AT       C       22.710  2024
4  AT       E        5.330  2021
5  AT       E        6.245  2022
6  AT       E        7.160  2023
7  AT       E       17.770  2024
8  AT       F        3.120  2021
9  AT       F        3.700  2022


In [158]:
df_ai_panel.to_csv('data_panel/ai_adopt.csv', index = False)

## ICT training (2020, 2022, 2024)

In [ ]:
train = pd.read_csv(path + 'estat_isoc_ske_ittn2.tsv/estat_isoc_ske_ittn2.tsv', sep='\t')
print(train)

      freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD   2012    2014   \
0                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,AT       :       :    
1                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,BA       :       :    
2                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,BE       :       :    
3                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,BG       :       :    
4                       A,GE10,C,E_ITSP2X_ITT2,PC_ENT,CY       :       :    
...                                                  ...      ...     ...   
16549                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,SE   37.50      : u   
16550                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,SI   57.14   49.93    
16551                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,SK   11.61   57.50    
16552                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,TR       :       :    
16553                A,GE10,S951,E_ITUST2,PC_ENT_CUSE,UK   46.48   43.08    

        2015    2016    2017    2018     2019    2020   2022    2024   
0  

In [34]:
train_split = train[r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD'].str.split(',', expand=True)
train = pd.concat([train, train_split], axis=1)
train.rename(columns={0: 'freq', 1: 'size_emp', 2: 'nace_r2',  3: 'indic_is',  4: 'unit', 5:  'geo_TIME_PERIOD'}, inplace=True)
train = train.drop(r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD', axis=1)
train.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
train.columns = train.columns.str.strip()
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16554 entries, 0 to 16553
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   2012      16554 non-null  object
 1   2014      16554 non-null  object
 2   2015      16554 non-null  object
 3   2016      16554 non-null  object
 4   2017      16554 non-null  object
 5   2018      16554 non-null  object
 6   2019      16554 non-null  object
 7   2020      16554 non-null  object
 8   2022      16554 non-null  object
 9   2024      16554 non-null  object
 10  freq      16554 non-null  object
 11  size_emp  16554 non-null  object
 12  nace_r2   16554 non-null  object
 13  indic_is  16554 non-null  object
 14  unit      16554 non-null  object
 15  geo       16554 non-null  object
dtypes: object(16)
memory usage: 2.0+ MB


In [35]:
years_take = ['2020', '2022', '2024']
data_temp  = train.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    # flag_col_name = f'flag_{year}'
    # data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
train_new = data_temp
print(train_new)

         2012    2014    2015    2016    2017    2018     2019    2020   2022  \
0          :       :       :   17.86   14.70   14.47     7.07    3.96   5.50    
1          :       :       :       :       :      : u      : u   2.90   7.08    
2          :       :       :      : u     : u     : u      : u     : u  8.87    
3          :       :       :      : u   1.49    1.30       : u     : u  1.80    
4          :       :       :    3.54   10.23   11.55    12.07   11.28   8.62    
...       ...     ...     ...     ...     ...     ...      ...     ...    ...   
16549  37.50      : u  27.27   50.00   40.91   31.03   33.33 b      :      :    
16550  57.14   49.93     : @C     : u    : @C    : @C     : @C      :      :    
16551  11.61   57.50       0   46.88   21.15       0        0       :      :    
16552      :       :      : u      :       :   39.56    28.05       :      :    
16553  46.48   43.08   55.27   49.64      : u     : u      : u      :      :    

         2024 freq size_emp

In [36]:
train_new.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
train_new.drop(years_take, axis='columns', inplace=True)
train_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16554 entries, 0 to 16553
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      16554 non-null  object 
 1   size_emp  16554 non-null  object 
 2   nace_r2   16554 non-null  object 
 3   indic_is  16554 non-null  object 
 4   unit      16554 non-null  object 
 5   geo       16554 non-null  object 
 6   num_2020  5499 non-null   float64
 7   num_2022  7014 non-null   float64
 8   num_2024  7069 non-null   float64
dtypes: float64(3), object(6)
memory usage: 1.1+ MB


In [37]:
# Filter the data 
train_new = train_new.copy()
train_new = train_new[
    (train_new['indic_is'] == 'E_ITT2') & # Enterprise provided training to their personnel to develop their ICT skills
    (train_new['unit'] == 'PC_ENT') # Percentage of enterprises 
]

train_new = train_new.drop(columns=['freq', 'size_emp', 'indic_is', 'unit'])
train_new = train_new.rename(columns={'num_2020' : 'tr_ict_2020',
                                    'num_2022' : 'tr_ict_2022',
                                    'num_2024' : 'tr_ict_2024'})
train_new = train_new[train_new['geo'].isin(EU_EFTA)]

print(train_new)

      nace_r2 geo  tr_ict_2020  tr_ict_2022  tr_ict_2024
239         C  AT        20.37        25.60        26.06
241         C  BE          NaN        33.89        40.12
242         C  BG         5.38         6.47         5.82
243         C  CY        19.69        20.51        17.41
244         C  CZ        27.60        23.76        28.35
...       ...  ..          ...          ...          ...
16425    S951  PT        63.29          NaN          NaN
16426    S951  RO        21.78        24.08        14.51
16428    S951  SE          NaN        53.33          NaN
16429    S951  SI          NaN          NaN        76.96
16430    S951  SK        18.33        23.08        18.18

[1403 rows x 5 columns]


In [38]:
train_new['nace_r2_1d'] = train_new['nace_r2'].map(nace_section_or_nan)
train_new.drop(columns=['nace_r2'], inplace=True) 
train_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
train_new = train_new.dropna()
print(train_new)

      geo  tr_ict_2020  tr_ict_2022  tr_ict_2024 nace_r2
239    AT        20.37        25.60        26.06       C
242    BG         5.38         6.47         5.82       C
243    CY        19.69        20.51        17.41       C
244    CZ        27.60        23.76        28.35       C
245    DE        26.58        27.82        28.73       C
...    ..          ...          ...          ...     ...
15233  PT        32.41        27.20        32.82       N
15234  RO         4.19         9.80        15.37       N
15236  SE        27.38        25.01        25.74       N
15237  SI        17.44        27.52        22.73       N
15238  SK        14.74         7.67        17.50       N

[182 rows x 5 columns]


In [ ]:
train_new['tr_ict_2021'] = (train_new['tr_ict_2020'] + train_new['tr_ict_2022']) /2
train_new['tr_ict_2023'] = (train_new['tr_ict_2022'] + train_new['tr_ict_2024']) /2
print(train_new.head())

    geo  tr_ict_2020  tr_ict_2022  tr_ict_2024 nace_r2  tr_ict_2021  \
239  AT        20.37        25.60        26.06       C       22.985   
242  BG         5.38         6.47         5.82       C        5.925   
243  CY        19.69        20.51        17.41       C       20.100   
244  CZ        27.60        23.76        28.35       C       25.680   
245  DE        26.58        27.82        28.73       C       27.200   

     tr_ict_2023  
239       25.830  
242        6.145  
243       18.960  
244       26.055  
245       28.275  


In [148]:
cols_to_melt = ['tr_ict_2020', 'tr_ict_2021', 'tr_ict_2022', 'tr_ict_2023', 'tr_ict_2024']

df_train_panel = train_new.melt(
    id_vars=['geo', 'nace_r2'],      
    value_vars=cols_to_melt,         
    var_name='year_raw',            
    value_name='training_ict'        
)

df_train_panel['year'] = df_train_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_train_panel.drop(columns=['year_raw'], inplace=True)
df_train_panel = df_train_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print("ICT Training Panel Format Ready:")
print(df_train_panel.head(10))

ICT Training Panel Format Ready:
  geo nace_r2  training_ict  year
0  AT       C        20.370  2020
1  AT       C        22.985  2021
2  AT       C        25.600  2022
3  AT       C        25.830  2023
4  AT       C        26.060  2024
5  AT       F         9.650  2020
6  AT       F         8.280  2021
7  AT       F         6.910  2022
8  AT       F         8.010  2023
9  AT       F         9.110  2024


In [149]:
df_train_panel.to_csv('data_panel/train.csv', index = False)

## ICT specialists (2020, 2022, 2024)

In [ ]:
ict_spec = pd.read_csv(path + 'estat_isoc_ske_itspen2.tsv/estat_isoc_ske_itspen2.tsv', sep='\t')
ict_spec_split = ict_spec[r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD'].str.split(',', expand=True)
ict_spec = pd.concat([ict_spec, ict_spec_split], axis=1)
ict_spec.rename(columns={0: 'freq', 1: 'size_emp', 2: 'nace_r2',  3: 'indic_is',  4: 'unit', 5:  'geo_TIME_PERIOD'}, inplace=True)
ict_spec = ict_spec.drop(r'freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD', axis=1)
ict_spec.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
ict_spec.columns = ict_spec.columns.str.strip()
ict_spec.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3343 entries, 0 to 3342
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   2012      3343 non-null   object
 1   2014      3343 non-null   object
 2   2015      3343 non-null   object
 3   2016      3343 non-null   object
 4   2017      3343 non-null   object
 5   2018      3343 non-null   object
 6   2019      3343 non-null   object
 7   2020      3343 non-null   object
 8   2022      3343 non-null   object
 9   2024      3343 non-null   object
 10  freq      3343 non-null   object
 11  size_emp  3343 non-null   object
 12  nace_r2   3343 non-null   object
 13  indic_is  3343 non-null   object
 14  unit      3343 non-null   object
 15  geo       3343 non-null   object
dtypes: object(16)
memory usage: 418.0+ KB


In [42]:
years_take = ['2020', '2022', '2024']
data_temp  = ict_spec.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    # flag_col_name = f'flag_{year}'
    # data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
ict_spec_new = data_temp
print(ict_spec_new)

        2012    2014     2015    2016    2017    2018    2019    2020  \
0         :       :        :       :       :       :       :       :    
1     35.12   31.01    31.44   35.22   30.53   28.03   29.00   28.74    
2         :       :        :       :       :   12.50   13.33   12.92    
3     32.85      : u      : u     : u     : u     : u  30.43      : u   
4     10.34   17.20    16.76   15.71   17.61   14.58   16.27   15.16    
...      ...     ...      ...     ...     ...     ...     ...     ...   
3338  75.00      : u   72.73   58.33   65.91      : u  39.29       :    
3339  71.43   83.33    83.33      : u    : @C    : @C     : u      :    
3340  41.96   57.50   100.00   93.75   28.85   28.85   31.82       :    
3341      :       :       : u      :       :   63.22   60.43       :    
3342  66.67   65.38    78.18   79.93      : u     : u     : u      :    

         2022    2024 freq size_emp nace_r2 indic_is         unit geo  \
0     11.22 b      :     A     GE10       C  E_ITS

In [43]:
ict_spec_new.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
ict_spec_new.drop(years_take, axis='columns', inplace=True)
ict_spec_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3343 entries, 0 to 3342
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      3343 non-null   object 
 1   size_emp  3343 non-null   object 
 2   nace_r2   3343 non-null   object 
 3   indic_is  3343 non-null   object 
 4   unit      3343 non-null   object 
 5   geo       3343 non-null   object 
 6   num_2020  1137 non-null   float64
 7   num_2022  1426 non-null   float64
 8   num_2024  1427 non-null   float64
dtypes: float64(3), object(6)
memory usage: 235.2+ KB


In [44]:
# Filter the data 
# E_ITSP2 - Enterprise employed ICT/IT specialists (reduced comparability with 2007)

ict_spec_new = ict_spec_new.copy()
ict_spec_new = ict_spec_new[
    (ict_spec_new['unit'] == 'PC_ENT') # Percentage of enterprises 
]

ict_spec_new = ict_spec_new.drop(columns=['freq', 'size_emp', 'indic_is', 'unit'])
ict_spec_new = ict_spec_new[ict_spec_new['geo'].isin(EU_EFTA)]

ict_spec_new['nace_r2_1d'] = ict_spec_new['nace_r2'].map(nace_section_or_nan)

ict_spec_new.drop(columns=['nace_r2'], inplace=True) 
ict_spec_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
ict_spec_new = ict_spec_new.dropna()
print(ict_spec_new)

     geo  num_2020  num_2022  num_2024 nace_r2
1     AT     28.74     28.37     30.02       C
4     BG     15.16     14.73     15.53       C
5     CY     14.23     16.29     14.93       C
6     CZ     21.66     21.42     23.05       C
7     DE     24.97     25.82     26.81       C
...   ..       ...       ...       ...     ...
3054  PT     29.51     29.67     29.37       N
3055  RO     14.31     11.35     15.56       N
3057  SE     17.58     17.64     11.41       N
3058  SI     10.39     14.76     13.80       N
3059  SK     16.90     13.37     13.68       N

[187 rows x 5 columns]


In [45]:
ict_spec_new = ict_spec_new.rename(columns={'num_2020': 'spec_ict_2020',
                                            'num_2022' : 'spec_ict_2022',
                                            'num_2024' : 'spec_ict_2024' })
print(ict_spec_new)

     geo  spec_ict_2020  spec_ict_2022  spec_ict_2024 nace_r2
1     AT          28.74          28.37          30.02       C
4     BG          15.16          14.73          15.53       C
5     CY          14.23          16.29          14.93       C
6     CZ          21.66          21.42          23.05       C
7     DE          24.97          25.82          26.81       C
...   ..            ...            ...            ...     ...
3054  PT          29.51          29.67          29.37       N
3055  RO          14.31          11.35          15.56       N
3057  SE          17.58          17.64          11.41       N
3058  SI          10.39          14.76          13.80       N
3059  SK          16.90          13.37          13.68       N

[187 rows x 5 columns]


In [57]:
ict_spec_new['spec_ict_2021'] = (ict_spec_new['spec_ict_2020'] + ict_spec_new['spec_ict_2022']) /2
ict_spec_new['spec_ict_2023'] = (ict_spec_new['spec_ict_2022'] + ict_spec_new['spec_ict_2024']) /2
print(ict_spec_new.head())

  geo  spec_ict_2020  spec_ict_2022  spec_ict_2024 nace_r2  spec_ict_2021  \
1  AT          28.74          28.37          30.02       C         28.555   
4  BG          15.16          14.73          15.53       C         14.945   
5  CY          14.23          16.29          14.93       C         15.260   
6  CZ          21.66          21.42          23.05       C         21.540   
7  DE          24.97          25.82          26.81       C         25.395   

   spec_ict_2023  
1         29.195  
4         15.130  
5         15.610  
6         22.235  
7         26.315  


In [146]:
cols_to_melt = ['spec_ict_2020', 'spec_ict_2021', 'spec_ict_2022', 'spec_ict_2023', 'spec_ict_2024']

df_ict_panel = ict_spec_new.melt(
    id_vars=['geo', 'nace_r2'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='spec_ict'           
)

df_ict_panel['year'] = df_ict_panel['year_raw'].str.extract(r'(\d+)').astype(int)

df_ict_panel.drop(columns=['year_raw'], inplace=True)

df_ict_panel = df_ict_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print("ICT Panel Format Ready:")
print(df_ict_panel.head(10))

ICT Panel Format Ready:
  geo nace_r2  spec_ict  year
0  AT       C    28.740  2020
1  AT       C    28.555  2021
2  AT       C    28.370  2022
3  AT       C    29.195  2023
4  AT       C    30.020  2024
5  AT       F     8.240  2020
6  AT       F     8.635  2021
7  AT       F     9.030  2022
8  AT       F     8.075  2023
9  AT       F     7.120  2024


In [147]:
df_ict_panel.to_csv('data_panel/ict_spec.csv', index = False)

## Wages 

### Nominal wage (wage per hour in EUR)

In [ ]:
lc = pd.read_csv(path + 'estat_lc_lci_lev.tsv/estat_lc_lci_lev.tsv', sep='\t')
lc_split = lc[r'freq,unit,lcstruct,nace_r2,geo\TIME_PERIOD'].str.split(',', expand=True)
lc = pd.concat([lc, lc_split], axis=1)
lc.rename(columns={0: 'freq', 1: 'unit', 2: 'lcstruct',  3: 'nace_r2',  4:  'geo_TIME_PERIOD'}, inplace=True)
lc = lc.drop(r'freq,unit,lcstruct,nace_r2,geo\TIME_PERIOD', axis=1)
lc.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
lc.columns = lc.columns.str.strip()
lc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11075 entries, 0 to 11074
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   2008      11075 non-null  object
 1   2012      11075 non-null  object
 2   2016      11075 non-null  object
 3   2020      11075 non-null  object
 4   2021      11075 non-null  object
 5   2022      11075 non-null  object
 6   2023      11075 non-null  object
 7   2024      11075 non-null  object
 8   freq      11075 non-null  object
 9   unit      11075 non-null  object
 10  lcstruct  11075 non-null  object
 11  nace_r2   11075 non-null  object
 12  geo       11075 non-null  object
dtypes: object(13)
memory usage: 1.1+ MB


In [60]:
years_take = ['2020', '2021', '2022', '2023', '2024']
data_temp  = lc.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    # flag_col_name = f'flag_{year}'
    # data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
lc_new = data_temp
print(lc_new)

        2008   2012   2016   2020   2021   2022    2023   2024 freq  \
0         :    2.1    2.7      :      :      :       :      :     A   
1      22.2   26.9   28.1   28.4   28.8   29.2   31.7 p  34.3     A   
2         :    6.3    6.7      :      :      :       :      :     A   
3         :   30.3   29.5   31.3   31.7     34    36.7   37.6     A   
4       3.3      5    5.7    8.1    8.8   10.3    11.4   12.2     A   
...      ...    ...    ...    ...    ...    ...     ...    ...  ...   
11070     :      :      :    8.4    4.8   19.8     8.2   18.9     A   
11071     :      :      :    6.6    9.5   15.3    12.2   13.3     A   
11072     :      :      :    0.1      4      1     3.8    2.2     A   
11073     :      :      :    4.1    3.6    4.2      14      7     A   
11074     :      :      :    4.1    5.5   12.7    11.1    6.4     A   

             unit   lcstruct nace_r2 geo  num_2020  num_2021  num_2022  \
0             EUR        D11       B  AL       NaN       NaN       NaN   

In [61]:
lc_new.drop(['2008', '2012','2016'], axis='columns', inplace=True)
lc_new.drop(years_take, axis='columns', inplace=True)
lc_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11075 entries, 0 to 11074
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      11075 non-null  object 
 1   unit      11075 non-null  object 
 2   lcstruct  11075 non-null  object 
 3   nace_r2   11075 non-null  object 
 4   geo       11075 non-null  object 
 5   num_2020  8758 non-null   float64
 6   num_2021  8802 non-null   float64
 7   num_2022  9272 non-null   float64
 8   num_2023  9461 non-null   float64
 9   num_2024  9211 non-null   float64
dtypes: float64(5), object(5)
memory usage: 865.4+ KB


In [62]:
print(lc_new)

      freq        unit   lcstruct nace_r2 geo  num_2020  num_2021  num_2022  \
0        A         EUR        D11       B  AL       NaN       NaN       NaN   
1        A         EUR        D11       B  AT      28.4      28.8      29.2   
2        A         EUR        D11       B  BA       NaN       NaN       NaN   
3        A         EUR        D11       B  BE      31.3      31.7      34.0   
4        A         EUR        D11       B  BG       8.1       8.8      10.3   
...    ...         ...        ...     ...  ..       ...       ...       ...   
11070    A  RT_PRE_NAC  D1_D4_MD5       S  RO       8.4       4.8      19.8   
11071    A  RT_PRE_NAC  D1_D4_MD5       S  RS       6.6       9.5      15.3   
11072    A  RT_PRE_NAC  D1_D4_MD5       S  SE       0.1       4.0       1.0   
11073    A  RT_PRE_NAC  D1_D4_MD5       S  SI       4.1       3.6       4.2   
11074    A  RT_PRE_NAC  D1_D4_MD5       S  SK       4.1       5.5      12.7   

       num_2023  num_2024  
0           NaN       N

In [63]:
# Filter the data 
# per employee in full-time equivalents, per hour

wg = lc_new.copy()
wg = wg[
    (wg['unit'] == 'EUR') &
    (wg['lcstruct'] == 'D11') # Wages and Salaries (total)
]

wg = wg.drop(columns=['freq', 'unit', 'lcstruct'])
wg = wg[wg['geo'].isin(EU_EFTA)]

print(wg)

    nace_r2 geo  num_2020  num_2021  num_2022  num_2023  num_2024
1         B  AT      28.4      28.8      29.2      31.7      34.3
3         B  BE      31.3      31.7      34.0      36.7      37.6
4         B  BG       8.1       8.8      10.3      11.4      12.2
5         B  CH       NaN       NaN       NaN       NaN       NaN
6         B  CY      13.6      14.9      15.0      16.4      17.1
..      ...  ..       ...       ...       ...       ...       ...
885       S  PT      10.2      10.9      10.6      11.5      12.2
886       S  RO       5.4       5.5       6.6       7.2       8.5
888       S  SE      23.5      24.7      23.9      23.3      23.8
889       S  SI      15.0      15.3      15.9      18.2      19.5
890       S  SK       6.8       7.4       7.8       8.6       8.9

[635 rows x 7 columns]


In [64]:
wg['nace_r2_1d'] = wg['nace_r2'].map(nace_section_or_nan)

wg.drop(columns=['nace_r2'], inplace=True) 
wg.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)

wg = wg.rename(columns={'num_2020': 'wg_n_2020',
                        'num_2021': 'wg_n_2021',
                        'num_2022' : 'wg_n_2022',
                        'num_2023' : 'wg_n_2023',
                        'num_2024' : 'wg_n_2024'})
print(wg)

    geo  wg_n_2020  wg_n_2021  wg_n_2022  wg_n_2023  wg_n_2024 nace_r2
1    AT       28.4       28.8       29.2       31.7       34.3       B
3    BE       31.3       31.7       34.0       36.7       37.6       B
4    BG        8.1        8.8       10.3       11.4       12.2       B
5    CH        NaN        NaN        NaN        NaN        NaN       B
6    CY       13.6       14.9       15.0       16.4       17.1       B
..   ..        ...        ...        ...        ...        ...     ...
885  PT       10.2       10.9       10.6       11.5       12.2       S
886  RO        5.4        5.5        6.6        7.2        8.5       S
888  SE       23.5       24.7       23.9       23.3       23.8       S
889  SI       15.0       15.3       15.9       18.2       19.5       S
890  SK        6.8        7.4        7.8        8.6        8.9       S

[635 rows x 7 columns]


### HICP (relative to 2015=100)

In [79]:
hicp = pd.read_csv(path + 'estat_prc_hicp_aind.tsv/estat_prc_hicp_aind.tsv', sep='\t')
print(hicp)

      freq,unit,coicop,geo\TIME_PERIOD 1996    1997    1998    1999    2000   \
0           A,CID_EA,TOT_X_NRG_FOOD,AT    :       :       :       :       :    
1           A,CID_EA,TOT_X_NRG_FOOD,BE    :       :       :       :       :    
2           A,CID_EA,TOT_X_NRG_FOOD,BG    :       :       :       :       :    
3           A,CID_EA,TOT_X_NRG_FOOD,CY    :       :       :       :       :    
4           A,CID_EA,TOT_X_NRG_FOOD,CZ    :       :       :       :       :    
...                                ...   ...     ...     ...     ...     ...   
35246         A,RCH_A_AVG,TOT_X_TBC,SI    :       :       :       :       :    
35247         A,RCH_A_AVG,TOT_X_TBC,SK    :     6.0     6.4    10.6    12.2    
35248         A,RCH_A_AVG,TOT_X_TBC,TR    :   85.0 d  82.6 d  61.4 d  52.4 d   
35249         A,RCH_A_AVG,TOT_X_TBC,UK    :     1.6     1.3     1.0     0.5    
35250         A,RCH_A_AVG,TOT_X_TBC,XK    :       :       :       :       :    

        2001    2002    2003   2004   .

In [80]:
hicp_split = hicp[r'freq,unit,coicop,geo\TIME_PERIOD'].str.split(',', expand=True)
hicp = pd.concat([hicp, hicp_split], axis=1)
hicp.rename(columns={0: 'freq', 1: 'unit', 2: 'coicop',  3:  'geo_TIME_PERIOD'}, inplace=True)
hicp = hicp.drop(r'freq,unit,coicop,geo\TIME_PERIOD', axis=1)
hicp.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
hicp.columns = hicp.columns.str.strip()
hicp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35251 entries, 0 to 35250
Data columns (total 33 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   1996    35251 non-null  object
 1   1997    35251 non-null  object
 2   1998    35251 non-null  object
 3   1999    35251 non-null  object
 4   2000    35251 non-null  object
 5   2001    35251 non-null  object
 6   2002    35251 non-null  object
 7   2003    35251 non-null  object
 8   2004    35251 non-null  object
 9   2005    35251 non-null  object
 10  2006    35251 non-null  object
 11  2007    35251 non-null  object
 12  2008    35251 non-null  object
 13  2009    35251 non-null  object
 14  2010    35251 non-null  object
 15  2011    35251 non-null  object
 16  2012    35251 non-null  object
 17  2013    35251 non-null  object
 18  2014    35251 non-null  object
 19  2015    35251 non-null  object
 20  2016    35251 non-null  object
 21  2017    35251 non-null  object
 22  2018    35251 non-null

In [81]:
years_take = ['2020', '2021', '2022', '2023', '2024']
data_temp  = hicp.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    # flag_col_name = f'flag_{year}'
    # data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
hicp_new = data_temp
print(hicp_new)


      1996    1997    1998    1999    2000    2001    2002    2003   2004  \
0       :       :       :       :       :       :    -0.4    -0.6   -0.3    
1       :       :       :       :       :       :    -0.3    -0.3   -0.6    
2       :       :       :       :       :       :     5.3     0.3    0.3    
3       :       :       :       :       :       :    -1.4     0.1   -2.0    
4       :       :       :       :       :       :     0.1    -1.2    0.1    
...    ...     ...     ...     ...     ...     ...     ...     ...    ...   
35246   :       :       :       :       :       :     7.1     5.3    3.4    
35247   :     6.0     6.4    10.6    12.2     7.2     3.0     8.1    7.2    
35248   :   85.0 d  82.6 d  61.4 d  52.4 d  56.6 d  46.8 d  24.7 d  9.7 d   
35249   :     1.6     1.3     1.0     0.5     1.1     1.2     1.2    1.2    
35250   :       :       :       :       :       :       :       :      :    

        2005  ...    2024 freq       unit          coicop geo num_2020  \
0

In [82]:
years_to_drop = [str(year) for year in range(1996, 2025)]
hicp_new = hicp_new.drop(columns=years_to_drop, errors='ignore')
hicp_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35251 entries, 0 to 35250
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      35251 non-null  object 
 1   unit      35251 non-null  object 
 2   coicop    35251 non-null  object 
 3   geo       35251 non-null  object 
 4   num_2020  27873 non-null  float64
 5   num_2021  29723 non-null  float64
 6   num_2022  31640 non-null  float64
 7   num_2023  31291 non-null  float64
 8   num_2024  29155 non-null  float64
dtypes: float64(5), object(4)
memory usage: 2.4+ MB


In [84]:
# Filter the data 
# annual average index

hicp_inx = hicp_new.copy()
hicp_inx = hicp_inx.reset_index(drop=True)
hicp_inx = hicp_inx[
    (hicp_inx['unit'] == 'INX_A_AVG') &
    (hicp_inx['coicop'] == 'CP00') 
]

hicp_inx = hicp_inx.drop(columns=['freq', 'unit', 'coicop'])
hicp_inx = hicp_inx[hicp_inx['geo'].isin(EU_EFTA)]

print(hicp_inx)

    geo  num_2020  num_2021  num_2022  num_2023  num_2024
233  AT    108.47    111.46    121.07    130.40    134.21
234  BE    108.23    111.71    123.26    126.07    131.52
235  BG    106.27    109.30    123.52    134.15    137.63
236  CH    100.56    101.04    103.74    106.10    107.25
237  CY     99.67    101.92    110.17    114.50    117.09
238  CZ    111.40    115.10    132.10    147.90    151.90
239  DE    105.80    109.20    118.70    125.90    129.00
240  DK    102.90    104.90    113.80    117.60    119.10
244  EE    109.80    114.72    137.03    149.52    155.10
246  EL    101.17    101.75    111.21    115.84    119.31
247  ES    103.91    107.04    115.95    119.89    123.33
251  FI    103.98    106.12    113.74    118.67    119.83
252  FR    105.50    107.68    114.04    120.50    123.29
253  HR    103.06    105.82    117.11    126.94    132.04
254  HU    113.15    119.04    137.22    160.59    166.56
255  IE    101.20    103.60    112.00    117.80    119.40
256  IS    103

In [85]:
hicp_inx = hicp_inx.rename(columns={'num_2020': 'hicp_inx_2020',
                        'num_2021': 'hicp_inx_2021',
                        'num_2022' : 'hicp_inx_2022',
                        'num_2023' : 'hicp_inx_2023',
                        'num_2024' : 'hicp_inx_2024'})

In [86]:
print(hicp_inx)

    geo  hicp_inx_2020  hicp_inx_2021  hicp_inx_2022  hicp_inx_2023  \
233  AT         108.47         111.46         121.07         130.40   
234  BE         108.23         111.71         123.26         126.07   
235  BG         106.27         109.30         123.52         134.15   
236  CH         100.56         101.04         103.74         106.10   
237  CY          99.67         101.92         110.17         114.50   
238  CZ         111.40         115.10         132.10         147.90   
239  DE         105.80         109.20         118.70         125.90   
240  DK         102.90         104.90         113.80         117.60   
244  EE         109.80         114.72         137.03         149.52   
246  EL         101.17         101.75         111.21         115.84   
247  ES         103.91         107.04         115.95         119.89   
251  FI         103.98         106.12         113.74         118.67   
252  FR         105.50         107.68         114.04         120.50   
253  H

### Calculate real wages

In [87]:
wage_countries = set(wg['geo'].unique())
hicp_countries = set(hicp_inx['geo'].unique())

missing_countries = wage_countries - hicp_countries

if len(missing_countries) > 0:
    print(f"WARNING: The following countries are in the Wage data but MISSING in HICP data:\n{missing_countries}")
else:
    print("All countries in Wage data have a matching HICP record.")

All countries in Wage data have a matching HICP record.


In [88]:
wg_merged = pd.merge(wg, hicp_inx, on='geo', how='left')

In [89]:
years = ['2020', '2021', '2022', '2023', '2024']

for year in years:
    # Define column names
    nom_col = f'wg_n_{year}'  
    hicp_col = f'hicp_inx_{year}'  
    real_col = f'wg_r_{year}'  
    
    # Apply formula
    wg_merged[real_col] = (wg_merged[nom_col] / wg_merged[hicp_col]) * 100

print(wg_merged)

    geo  wg_n_2020  wg_n_2021  wg_n_2022  wg_n_2023  wg_n_2024 nace_r2  \
0    AT       28.4       28.8       29.2       31.7       34.3       B   
1    BE       31.3       31.7       34.0       36.7       37.6       B   
2    BG        8.1        8.8       10.3       11.4       12.2       B   
3    CH        NaN        NaN        NaN        NaN        NaN       B   
4    CY       13.6       14.9       15.0       16.4       17.1       B   
..   ..        ...        ...        ...        ...        ...     ...   
630  PT       10.2       10.9       10.6       11.5       12.2       S   
631  RO        5.4        5.5        6.6        7.2        8.5       S   
632  SE       23.5       24.7       23.9       23.3       23.8       S   
633  SI       15.0       15.3       15.9       18.2       19.5       S   
634  SK        6.8        7.4        7.8        8.6        8.9       S   

     hicp_inx_2020  hicp_inx_2021  hicp_inx_2022  hicp_inx_2023  \
0           108.47         111.46         12

In [90]:
wg_merged = wg_merged.drop(['wg_n_2020', 'wg_n_2021', 'wg_n_2022', 'wg_n_2023', 'wg_n_2024',
                            'hicp_inx_2020', 'hicp_inx_2021', 'hicp_inx_2022', 'hicp_inx_2023', 'hicp_inx_2024'], axis='columns')
wg_merged = wg_merged.dropna()
print(wg_merged)

    geo nace_r2  wg_r_2020  wg_r_2021  wg_r_2022  wg_r_2023  wg_r_2024
0    AT       B  26.182355  25.838866  24.118279  24.309816  25.556963
1    BE       B  28.919893  28.377048  27.583969  29.110811  28.588808
2    BG       B   7.622095   8.051235   8.338731   8.497950   8.864346
4    CY       B  13.645029  14.619309  13.615322  14.323144  14.604151
5    CZ       B  10.053860  10.251955  10.068130  10.074375   9.743252
..   ..     ...        ...        ...        ...        ...        ...
630  PT       S   9.847461  10.425634   9.378041   9.665490   9.987720
631  RO       S   4.879371   4.773891   5.113901   5.083310   5.670069
632  SE       S  21.834061  22.354964  20.018427  18.427713  18.452473
633  SI       S  14.310246  14.303076  13.596716  14.515872  15.249863
634  SK       S   6.269014   6.634986   6.237505   6.196412   6.216386

[441 rows x 7 columns]


In [144]:
wg_panel = wg_merged.melt(
    id_vars=['geo', 'nace_r2'],    
    value_vars=['wg_r_2020', 'wg_r_2021', 'wg_r_2022', 'wg_r_2023', 'wg_r_2024'], #
    var_name='year_raw',            
    value_name='real_wage'          
)

wg_panel['year'] = wg_panel['year_raw'].str.extract(r'(\d+)').astype(int)
wg_panel.drop(columns=['year_raw'], inplace=True)
wg_panel = wg_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print("Panel Format Ready:")
print(wg_panel.head(10))

Panel Format Ready:
  geo nace_r2  real_wage  year
0  AT       B  26.182355  2020
1  AT       B  25.838866  2021
2  AT       B  24.118279  2022
3  AT       B  24.309816  2023
4  AT       B  25.556963  2024
5  AT       C  27.104268  2020
6  AT       C  26.736049  2021
7  AT       C  26.183200  2022
8  AT       C  26.073620  2023
9  AT       C  27.047165  2024


In [145]:
wg_panel.to_csv('data_panel/wage.csv', index = False)

## Productivity (2021-2024)

In [107]:
product = pd.read_csv(path + 'estat_sbs_sc_ovw.tsv/estat_sbs_sc_ovw.tsv', sep='\t')

In [108]:
print(product)

        freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD    2021     2022   \
0                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,AL    5.11     5.07    
1                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,AT   49.73    54.31    
2                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BA    7.26     8.89    
3                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BE   45.15    53.13    
4                      A,AVG_EXPN_SAL_BEN_TEUR,B,0-9,BG   12.48    11.54    
...                                                 ...      ...      ...   
1410270                       A,WAGE_MEUR,S960,TOTAL,RO  199.03   211.74    
1410271                       A,WAGE_MEUR,S960,TOTAL,RS  43.30 b   48.03    
1410272                       A,WAGE_MEUR,S960,TOTAL,SE  863.44   891.76    
1410273                       A,WAGE_MEUR,S960,TOTAL,SI   65.60    71.97    
1410274                       A,WAGE_MEUR,S960,TOTAL,SK   46.87    56.30    

           2023  2024   
0          5.79     :   
1         55.31     :   


In [109]:
product_split = product[r'freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD'].str.split(',', expand=True)
product = pd.concat([product, product_split], axis=1)
product.rename(columns={0: 'freq', 1: 'indic_sbs',  2: 'nace_r2',  3: 'size_emp', 4: 'geo_TIME_PERIOD'}, inplace=True)
product = product.drop(r'freq,indic_sbs,nace_r2,size_emp,geo\TIME_PERIOD', axis=1)
product.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
product.columns = product.columns.str.strip()
product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1410275 entries, 0 to 1410274
Data columns (total 9 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   2021       1410275 non-null  object
 1   2022       1410275 non-null  object
 2   2023       1410275 non-null  object
 3   2024       1410275 non-null  object
 4   freq       1410275 non-null  object
 5   indic_sbs  1410275 non-null  object
 6   nace_r2    1410275 non-null  object
 7   size_emp   1410275 non-null  object
 8   geo        1410275 non-null  object
dtypes: object(9)
memory usage: 96.8+ MB


In [110]:
years_take = ['2021', '2022', '2023', '2024']
data_temp  = product.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    # flag_col_name = f'flag_{year}'
    # data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
product_new = data_temp
print(product_new)

            2021     2022     2023 2024 freq              indic_sbs nace_r2  \
0          5.11     5.07     5.79    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
1         49.73    54.31    55.31    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
2          7.26     8.89    10.07    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
3         45.15    53.13    51.45    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
4         12.48    11.54    12.77    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
...          ...      ...      ...  ...  ...                    ...     ...   
1410270  199.03   211.74   256.91    :     A              WAGE_MEUR    S960   
1410271  43.30 b   48.03    57.22    :     A              WAGE_MEUR    S960   
1410272  863.44   891.76   805.37    :     A              WAGE_MEUR    S960   
1410273   65.60    71.97    81.64    :     A              WAGE_MEUR    S960   
1410274   46.87    56.30    65.11    :     A              WAGE_MEUR    S960   

        size_emp geo  num_2021  num_2022  num_2023 

In [ ]:
product_new['nace_r2_1d'] = product_new['nace_r2'].map(nace_section_or_nan)
product_new.drop(columns=['nace_r2'], inplace=True) 
product_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
product_new = product_new.drop(['2021', '2022', '2023', '2024'], axis='columns')
product_new = product_new[product_new['geo'].isin(EU_EFTA)]

In [114]:
print(product_new)

        freq              indic_sbs size_emp geo  num_2021  num_2022  \
1          A  AVG_EXPN_SAL_BEN_TEUR      0-9  AT     49.73     54.31   
3          A  AVG_EXPN_SAL_BEN_TEUR      0-9  BE     45.15     53.13   
4          A  AVG_EXPN_SAL_BEN_TEUR      0-9  BG     12.48     11.54   
5          A  AVG_EXPN_SAL_BEN_TEUR      0-9  CY       NaN       NaN   
6          A  AVG_EXPN_SAL_BEN_TEUR      0-9  CZ     19.62     22.49   
...      ...                    ...      ...  ..       ...       ...   
1410269    A              WAGE_MEUR    TOTAL  PT    271.46    308.74   
1410270    A              WAGE_MEUR    TOTAL  RO    199.03    211.74   
1410272    A              WAGE_MEUR    TOTAL  SE    863.44    891.76   
1410273    A              WAGE_MEUR    TOTAL  SI     65.60     71.97   
1410274    A              WAGE_MEUR    TOTAL  SK     46.87     56.30   

         num_2023  num_2024 nace_r2  
1           55.31       NaN       B  
3           51.45       NaN       B  
4           12.77    

In [ ]:
# Filter 
product_indicators = ['AV_MEUR', 'EMP_NR']
product_filter = product_new[
            (product_new['indic_sbs'].isin(product_indicators)) & 
            (product_new['size_emp'] == 'TOTAL')
            ].copy()


df_long = product_filter.melt(
    id_vars=['geo', 'nace_r2', 'indic_sbs'], 
    value_vars=['num_2021', 'num_2022', 'num_2023', 'num_2024'], 
    var_name='year', 
    value_name='value'
)
df_long['year'] = df_long['year'].str.replace('num_', '').astype(int)

product_final = df_long.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='indic_sbs', 
    values='value'
).reset_index()

product_final['prodct_calc'] = (product_final['AV_MEUR'] * 1000) / product_final['EMP_NR']

# Inspect the result for 2024
print(product_final[product_final['year'] == 2024].head())

indic_sbs geo nace_r2  year  AV_MEUR    EMP_NR  prodct_calc
3          AT       B  2024      NaN    7561.0          NaN
7          AT       C  2024      NaN  728325.0          NaN
11         AT       D  2024      NaN   39906.0          NaN
15         AT       E  2024      NaN   23306.0          NaN
19         AT       F  2024      NaN  353909.0          NaN


In [ ]:
product_final = product_final.dropna()
product_final = product_final.drop(['AV_MEUR', 'EMP_NR'], axis='columns')

In [120]:
product_final.to_csv('data_panel/product.csv', index = False)

## Firm size

In [121]:
print(product)

            2021     2022     2023 2024 freq              indic_sbs nace_r2  \
0          5.11     5.07     5.79    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
1         49.73    54.31    55.31    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
2          7.26     8.89    10.07    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
3         45.15    53.13    51.45    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
4         12.48    11.54    12.77    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
...          ...      ...      ...  ...  ...                    ...     ...   
1410270  199.03   211.74   256.91    :     A              WAGE_MEUR    S960   
1410271  43.30 b   48.03    57.22    :     A              WAGE_MEUR    S960   
1410272  863.44   891.76   805.37    :     A              WAGE_MEUR    S960   
1410273   65.60    71.97    81.64    :     A              WAGE_MEUR    S960   
1410274   46.87    56.30    65.11    :     A              WAGE_MEUR    S960   

        size_emp geo  
0            0-9  AL  
1    

In [122]:
years_take = ['2021', '2022', '2023', '2024']
data_temp  = product.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    # flag_col_name = f'flag_{year}'
    # data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
firm_new = data_temp
print(firm_new)

            2021     2022     2023 2024 freq              indic_sbs nace_r2  \
0          5.11     5.07     5.79    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
1         49.73    54.31    55.31    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
2          7.26     8.89    10.07    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
3         45.15    53.13    51.45    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
4         12.48    11.54    12.77    :     A  AVG_EXPN_SAL_BEN_TEUR       B   
...          ...      ...      ...  ...  ...                    ...     ...   
1410270  199.03   211.74   256.91    :     A              WAGE_MEUR    S960   
1410271  43.30 b   48.03    57.22    :     A              WAGE_MEUR    S960   
1410272  863.44   891.76   805.37    :     A              WAGE_MEUR    S960   
1410273   65.60    71.97    81.64    :     A              WAGE_MEUR    S960   
1410274   46.87    56.30    65.11    :     A              WAGE_MEUR    S960   

        size_emp geo  num_2021  num_2022  num_2023 

In [123]:
firm_new['nace_r2_1d'] = firm_new['nace_r2'].map(nace_section_or_nan)
firm_new.drop(columns=['nace_r2'], inplace=True) 
firm_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
firm_new = firm_new.drop(['2021', '2022', '2023', '2024'], axis='columns')
firm_new = firm_new[firm_new['geo'].isin(EU_EFTA)]

In [125]:
# --- FSI Calculation (2021-2024) ---

# 1. Filter for Indicator and Size Classes (Same as before)
df_emp = firm_new[firm_new['indic_sbs'] == 'EMP_NR'].copy()
df_fsi = df_emp[df_emp['size_emp'].isin(['GE250', 'TOTAL'])].copy()

# 2. MELT: Transform year columns into rows
# This combines num_2021, num_2022, etc. into a single 'year' column
df_melted = df_fsi.melt(
    id_vars=['geo', 'nace_r2', 'size_emp'],      # Columns to keep fixed
    value_vars=['num_2021', 'num_2022', 'num_2023', 'num_2024'], # Columns to stack
    var_name='year_raw',
    value_name='emp_value'
)

df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d+)').astype(int)

df_pivoted = df_melted.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='size_emp',
    values='emp_value',
    aggfunc='sum'
).reset_index()

df_pivoted.rename(columns={'GE250': 'Emp_250_plus', 'TOTAL': 'Emp_Total'}, inplace=True)

df_pivoted['FSI'] = (df_pivoted['Emp_250_plus'] / df_pivoted['Emp_Total']) * 100

df_fsi_final = df_pivoted[['geo', 'nace_r2', 'year', 'FSI']].copy()

print("FSI Calculation Head (All Years):")
print(df_fsi_final.head())

FSI Calculation Head (All Years):
size_emp geo nace_r2  year        FSI
0         AT       B  2021   0.000000
1         AT       B  2022   0.000000
2         AT       B  2023   0.000000
3         AT       B  2024  39.214390
4         AT       C  2021  55.824158


In [126]:
df_fsi_final.to_csv('data_panel/fsi.csv', index = False)

## Education

In [132]:
educ = pd.read_csv(path + 'estat_edat_lfs_9910.tsv/estat_edat_lfs_9910.tsv', sep='\t')
educ_split = educ[r'freq,unit,nace_r2,isced11,age,sex,geo\TIME_PERIOD'].str.split(',', expand=True)
educ = pd.concat([educ, educ_split], axis=1)
educ.rename(columns={0: 'freq', 1: 'unit', 2: 'nace_r2',  3: 'isced11',  4: 'age', 5: 'sex', 6:  'geo_TIME_PERIOD'}, inplace=True)
educ = educ.drop(r'freq,unit,nace_r2,isced11,age,sex,geo\TIME_PERIOD', axis=1)
educ.rename(columns={"geo_TIME_PERIOD": "geo"}, inplace=True)
educ.columns = educ.columns.str.strip()
educ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241730 entries, 0 to 241729
Data columns (total 24 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   2008     241730 non-null  object
 1   2009     241730 non-null  object
 2   2010     241730 non-null  object
 3   2011     241730 non-null  object
 4   2012     241730 non-null  object
 5   2013     241730 non-null  object
 6   2014     241730 non-null  object
 7   2015     241730 non-null  object
 8   2016     241730 non-null  object
 9   2017     241730 non-null  object
 10  2018     241730 non-null  object
 11  2019     241730 non-null  object
 12  2020     241730 non-null  object
 13  2021     241730 non-null  object
 14  2022     241730 non-null  object
 15  2023     241730 non-null  object
 16  2024     241730 non-null  object
 17  freq     241730 non-null  object
 18  unit     241730 non-null  object
 19  nace_r2  241730 non-null  object
 20  isced11  241730 non-null  object
 21  age      2

In [134]:
years_take = ['2020', '2021', '2022', '2023', '2024']
data_temp  = educ.copy()

pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)?\s*[: ]*\s*([A-Za-z]+)?\s*$')

for year in years_take:
    column_series = data_temp[year].astype(str).str.strip() 

    extracted_df = column_series.str.extract(pattern, expand=True)
    # print(extracted_df)

    num_col_name = f'num_{year}'
    data_temp[num_col_name] = pd.to_numeric(extracted_df[0], errors='coerce') 
    # print(ain2[num_col_name])
    
    # flag_col_name = f'flag_{year}'
    # data_temp[flag_col_name] = extracted_df[1].fillna('NaN') 
    # # print(ain2[flag_col_name])
    # print('#########################')
# Display the resulting DataFrame
educ_new = data_temp
print(educ_new)

        2008 2009     2010    2011    2012    2013     2014    2015    2016  \
0        : u  : u      : u     : u     : u     : u     : bu     : u     : u   
1         :    :        :       :       :       :        :       :       :    
2       : bu  : u      : u     : u     : u     : u     : bu     : u     : u   
3        : u  : u     : bu    : bu     : u     : u     : bu     : u     : u   
4        : u  : u  68.1 bu  60.4 u  46.9 u  58.9 u  61.0 bu  36.8 u  33.1 u   
...      ...  ...      ...     ...     ...     ...      ...     ...     ...   
241725   : u  : u      : u     : u     : u     : u     : bu     : u     : u   
241726    :    :        :       :       :       :        :       :       :    
241727   : u   :        :     : bu     : u      :      : bu     : u     : u   
241728    :   : u      : u  72.4 u  74.7 u  36.0 u       :      : u     : u   
241729  : bu  : u     : bu    : bu     : u     : u     : bu     : u     : u   

          2017  ... nace_r2 isced11     age sex geo

In [135]:
educ_new.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
educ_new.drop(years_take, axis='columns', inplace=True)
educ_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241730 entries, 0 to 241729
Data columns (total 12 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   freq      241730 non-null  object 
 1   unit      241730 non-null  object 
 2   nace_r2   241730 non-null  object 
 3   isced11   241730 non-null  object 
 4   age       241730 non-null  object 
 5   sex       241730 non-null  object 
 6   geo       241730 non-null  object 
 7   num_2020  114497 non-null  float64
 8   num_2021  153586 non-null  float64
 9   num_2022  154677 non-null  float64
 10  num_2023  151890 non-null  float64
 11  num_2024  151934 non-null  float64
dtypes: float64(5), object(7)
memory usage: 22.1+ MB


In [136]:
# Filter the data 
# unit - PC, percent

educ = educ_new.copy()
educ = educ[
    (educ['sex'] == 'T')& # for all genders
    (educ['age'] == 'Y18-69')& 
    (educ['isced11'] == 'ED5-8')   # 5-8: Tertiary education (levels 5-8)
]

educ = educ.drop(columns=['freq', 'unit', 'age', 'sex', 'isced11'])
educ = educ[educ['geo'].isin(EU_EFTA)]
educ = educ.dropna()

print(educ)

       nace_r2 geo  num_2020  num_2021  num_2022  num_2023  num_2024
9664         A  AT      30.4      22.2      24.4      22.6      33.8
9666         A  BE      19.8      28.7      36.2      36.0      41.1
9667         A  BG       8.1       7.5       6.7       8.6      12.7
9668         A  CH      21.7      23.4      22.9      23.9      29.5
9669         A  CY      15.9      21.5      21.4      18.9      17.3
...        ...  ..       ...       ...       ...       ...       ...
240702       U  CH      81.3      75.4      70.1      75.9      81.2
240703       U  CY      53.0      50.5      48.0      63.4      56.1
240713       U  FR      79.7      77.9      79.5      75.4      75.5
240718       U  IT      52.1      60.9      60.8      63.6      64.3
240720       U  LU      89.4      90.1      94.0      94.1      93.6

[573 rows x 7 columns]


In [137]:
educ.to_csv('data_panel/educ.csv', index = False)

## High skills

In [138]:
occup = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/occup.csv')

# Filter 
# unit - Ths_per, thousandpersons
occup = occup.copy()
occup = occup[
    (occup['sex'] == 'T')& # for all genders
    (occup['age'] == 'Y20-64') # From 20 to 64 years
]

occup = occup.drop(columns=['freq', 'unit', 'age', 'sex', 'flag_2020', 'flag_2021', 'flag_2022', 'flag_2023', 'flag_2024'])
occup = occup[occup['geo'].isin(EU_EFTA)]
print(occup)

      nace_r2 isco08 geo  num_2020  num_2021  num_2022  num_2023  num_2024
18030       A    NRP  CH       NaN       1.6       2.1       1.4       1.7
18031       A    NRP  DE       NaN       NaN       NaN       NaN       NaN
18032       A    NRP  DK       NaN       NaN       NaN       NaN       NaN
18035       A    NRP  FI       NaN       NaN       NaN       NaN       NaN
18036       A    NRP  FR       NaN       6.2       NaN       NaN       NaN
...       ...    ...  ..       ...       ...       ...       ...       ...
27360       U  TOTAL  PT       NaN       NaN       NaN       NaN       NaN
27361       U  TOTAL  RO       NaN       NaN       NaN       NaN       NaN
27363       U  TOTAL  SE       NaN       NaN       NaN       NaN       NaN
27364       U  TOTAL  SI       NaN       NaN       NaN       NaN       NaN
27365       U  TOTAL  SK       NaN       NaN       NaN       NaN       NaN

[7370 rows x 8 columns]


In [139]:
occup['nace_r2_1d'] = occup['nace_r2'].map(nace_section_or_nan)

occup.drop(columns=['nace_r2'], inplace=True) 
occup.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
# print(occup)

occup = occup.dropna()
occup.rename(columns={'num_2020': 'occup_2020', 'num_2021' : 'occup_2021', 'num_2022' : 'occup_2022', 'num_2023' : 'occup_2023','num_2024' : 'occup_2024' }, inplace=True)
print(occup)

      isco08 geo  occup_2020  occup_2021  occup_2022  occup_2023  occup_2024  \
18076    OC1  CZ         3.2         4.1         4.6         5.0         4.6   
18082    OC1  ES        14.4        13.2        16.5        17.3        17.1   
18085    OC1  FR        17.9        26.0        31.8        32.5        28.4   
18086    OC1  HR         2.1         1.9         2.0         2.7         3.0   
18087    OC1  HU         5.7         5.9         7.2         6.6         7.1   
...      ...  ..         ...         ...         ...         ...         ...   
27338  TOTAL  DK         2.7         2.7         2.9         3.3         2.2   
27342  TOTAL  ES         4.5         4.1         3.4         4.1         5.9   
27345  TOTAL  FR        20.1        16.3        14.5        18.4        19.9   
27350  TOTAL  IT        16.4        15.7        18.1        16.6        16.4   
27352  TOTAL  LU        16.2        19.0        20.0        21.2        26.0   

      nace_r2  
18076       A  
18082  

In [141]:
high_skill_codes = ['OC1', 'OC2', 'OC3']
total_code = 'TOTAL'
relevant_codes = high_skill_codes + [total_code]

occup_filtered = occup[occup['isco08'].isin(relevant_codes)].copy()

df_melted = occup_filtered.melt(
    id_vars=['geo', 'nace_r2', 'isco08'],          
    value_vars=['occup_2020','occup_2021', 'occup_2022', 'occup_2023', 'occup_2024'], 
    var_name='year_raw',
    value_name='emp_value'
)

df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d+)').astype(int)

occup_pivoted = df_melted.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='isco08',
    values='emp_value',
    aggfunc='sum'
).reset_index()


occup_pivoted.rename(columns={
    'OC1': 'OC1_Managers',
    'OC2': 'OC2_Professionals',
    'OC3': 'OC3_Technicians',
    'TOTAL': 'TOTAL_Occupied'
}, inplace=True)


occup_pivoted['total_high_skill'] = (
    occup_pivoted['OC1_Managers'] + 
    occup_pivoted['OC2_Professionals'] + 
    occup_pivoted['OC3_Technicians']
)



occup_pivoted['share_high_skill'] = (
    occup_pivoted['total_high_skill'] / occup_pivoted['TOTAL_Occupied']
) * 100



share_high_skill_df = occup_pivoted[['geo', 'nace_r2', 'year', 'share_high_skill']].copy()

share_high_skill_df = share_high_skill_df.dropna()

print("High Skill Share Calculation Head (Panel):")
print(share_high_skill_df.head())

High Skill Share Calculation Head (Panel):
isco08 geo nace_r2  year  share_high_skill
10      AT       C  2020         35.562831
11      AT       C  2021         36.579294
12      AT       C  2022         37.320574
13      AT       C  2023         38.711970
14      AT       C  2024         40.126825


In [142]:
share_high_skill_df.to_csv('data_panel/share_high_skill.csv', index = False)

## Panel main dataset 

In [182]:
# df_ai_panel
# df_train_panel
# df_ict_panel
# wg_panel
# product_final
# df_fsi_final
# share_high_skill_df

# Filter the base Wage panel to start from 2021
df_master = wg_panel.copy()
df_master = pd.merge(df_master, df_ai_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_ict_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_train_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, product_final, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_fsi_final, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, share_high_skill_df, on=['geo', 'nace_r2', 'year'], how='left')


print("\n--- MASTER PANEL DATASET (2021-2024) ---")
print(df_master['year'].value_counts().sort_index()) # Verifies only 2021, 2022, 2023, 2024 exist
print(df_master)


--- MASTER PANEL DATASET (2021-2024) ---
year
2020    441
2021    441
2022    441
2023    441
2024    441
Name: count, dtype: int64
     geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0     AT       B  26.182355  2020          NaN       NaN           NaN   
1     AT       B  25.838866  2021          NaN       NaN           NaN   
2     AT       B  24.118279  2022          NaN       NaN           NaN   
3     AT       B  24.309816  2023          NaN       NaN           NaN   
4     AT       B  25.556963  2024          NaN       NaN           NaN   
...   ..     ...        ...   ...          ...       ...           ...   
2200  SK       S   6.269014  2020          NaN       NaN           NaN   
2201  SK       S   6.634986  2021          NaN       NaN           NaN   
2202  SK       S   6.237505  2022          NaN       NaN           NaN   
2203  SK       S   6.196412  2023          NaN       NaN           NaN   
2204  SK       S   6.216386  2024          NaN       

In [183]:
# Check unique values in your BASE dataset (Wages)
print("--- WAGE NACE Codes ---")
print(wg_panel['nace_r2'].unique())
print(f"Type: {wg_panel['nace_r2'].dtype}")

# Check unique values in your AI dataset
print("\n--- AI NACE Codes ---")
print(df_ai_panel['nace_r2'].unique())
print(f"Type: {df_ai_panel['nace_r2'].dtype}")

# Check Year types (Must be identical)
print("\n--- Year Types ---")
print(f"Wage Year Type: {wg_panel['year'].dtype}")
print(f"AI Year Type: {df_ai_panel['year'].dtype}")

--- WAGE NACE Codes ---
['B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'M' 'N' 'P' 'Q' 'R' 'S']
Type: object

--- AI NACE Codes ---
['C' 'E' 'F' 'G' 'H' 'I' 'J' 'M' 'N']
Type: object

--- Year Types ---
Wage Year Type: int32
AI Year Type: int32


In [184]:
wg_panel['nace_r2'] = wg_panel['nace_r2'].astype(str).str.strip()
df_ai_panel['nace_r2'] = df_ai_panel['nace_r2'].astype(str).str.strip()
df_ict_panel['nace_r2'] = df_ict_panel['nace_r2'].astype(str).str.strip()
df_train_panel['nace_r2'] = df_train_panel['nace_r2'].astype(str).str.strip()
product_final['nace_r2'] = product_final['nace_r2'].astype(str).str.strip()
df_fsi_final['nace_r2'] = df_fsi_final['nace_r2'].astype(str).str.strip()
share_high_skill_df['nace_r2'] = share_high_skill_df['nace_r2'].astype(str).str.strip()

In [185]:
df_master = wg_panel[wg_panel['year'] >= 2021].copy()

In [186]:
df_master = pd.merge(df_master, df_ai_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_ict_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_train_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, product_final, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_fsi_final, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, share_high_skill_df, on=['geo', 'nace_r2', 'year'], how='left')

In [187]:
check_sector = df_master[df_master['nace_r2'] == 'C']
print(f"Total rows in Master: {len(df_master)}")
print(f"Rows for Sector C (Manufacturing): {len(check_sector)}")
print(f"Non-missing AI values in Sector C: {check_sector['ai_adoption'].count()}")

# Show a success sample
print("\n--- Success Sample (Sector C) ---")
print(check_sector[['geo', 'year', 'ai_adoption']].head())

Total rows in Master: 1764
Rows for Sector C (Manufacturing): 112
Non-missing AI values in Sector C: 108

--- Success Sample (Sector C) ---
   geo  year  ai_adoption
4   AT  2021         9.61
5   AT  2022        10.96
6   AT  2023        12.31
7   AT  2024        22.71
68  BE  2021        10.42


In [188]:
# Drop rows where AI data is missing (removes sectors B, P, Q, etc.)
df_clean = df_master.dropna(subset=['ai_adoption']).copy()
print(f"\nFinal Valid Observations for Model: {len(df_clean)}")
print(f"Sectors included in study: {df_clean['nace_r2'].unique()}")


Final Valid Observations for Model: 912
Sectors included in study: ['C' 'E' 'F' 'G' 'H' 'I' 'J' 'M' 'N']


In [189]:
df_clean['prod_lag'] = df_clean.groupby(['geo', 'nace_r2'])['prodct_calc'].shift(1)
df_clean['log_prod_lag'] = np.log(df_clean['prod_lag'])

In [190]:
print("\n--- Number of Missing Data by Year  ---")
missing_by_year = df_clean.groupby('year').apply(lambda x: x.isnull().sum())

cols_with_missing = missing_by_year.columns[missing_by_year.sum() > 0]
print(missing_by_year[cols_with_missing])

print("\n--- Percentage of Missing Data by Year (%) ---")
missing_percent = df_clean.groupby('year').apply(lambda x: (x.isnull().sum() / len(x)) * 100).round(1)
print(missing_percent[cols_with_missing])


--- Number of Missing Data by Year  ---
      spec_ict  training_ict  prodct_calc  share_high_skill  prod_lag  \
year                                                                    
2021        52            55            0                55       228   
2022        52            55            0                55         0   
2023        52            55            0                55         0   
2024        52            55          228                55         0   

      log_prod_lag  
year                
2021           228  
2022             0  
2023             0  
2024             0  

--- Percentage of Missing Data by Year (%) ---
      spec_ict  training_ict  prodct_calc  share_high_skill  prod_lag  \
year                                                                    
2021      22.8          24.1          0.0              24.1     100.0   
2022      22.8          24.1          0.0              24.1       0.0   
2023      22.8          24.1          0.0             

C:\Users\ydmar\AppData\Local\Temp\ipykernel_16456\949365043.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_by_year = df_clean.groupby('year').apply(lambda x: x.isnull().sum())
C:\Users\ydmar\AppData\Local\Temp\ipykernel_16456\949365043.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_percent = df_clean.groupby('year').apply(lambda x: (x.isnull().sum() / len(x)) * 100).round(1)


In [192]:
required_vars = [
    'real_wage', 
    'ai_adoption', 
    'spec_ict', 
    'training_ict', 
    'share_high_skill', 
    'log_prod_lag',    
    'FSI'
]

df_final = df_clean.dropna(subset=required_vars).copy()
print(f"Original Row Count: {len(df_clean)}")
print(f"Final Valid Row Count: {len(df_final)}")
print("\nYears remaining in study:", df_final['year'].unique())
print("Sectors remaining:", df_final['nace_r2'].unique())

Original Row Count: 912
Final Valid Row Count: 405

Years remaining in study: [2022 2023 2024]
Sectors remaining: ['C' 'F' 'G' 'H' 'N' 'I' 'J']


In [194]:
df_final.to_csv('data_panel/panel_master.csv', index = False )